# Pyomo Homework 2

```{warning}
**This is a draft assignment. It is still being updated for Fall 2026.**
```

In [ ]:
# This code cell installs packages on Colab

import sys

if "google.colab" in sys.modules:
    !wget "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/helper.py"
    import helper

    helper.easy_install()
else:
    sys.path.insert(0, "../")
    import helper
helper.set_plotting_style()

In [ ]:
## IMPORT LIBRARIES
import pyomo.environ as pyo
import pandas as pd

Special thanks to the Pyomo team for creating these exercises as part of their excellent PyomoFest workshop.

## 1 Pyomo Fundamentals

### 1.1 Knapsack example

You want to fill a knapsack (a.k.a. bag). You can choose from a hammer, wrench, screwdriver, and towel. Each item has a different weight and value. You want to maximize the value (benefit) of the collection of items constrained by a total weight limit. Let's formulate this as an optimization problem.

**Sets**

$$\mathcal{A} = \{\text{hammer},~\text{wrench},~\text{screwdriver},~\text{towel} \}$$  

**Parameters (Data)**

Let $b_i$ and $w_i$ represent the benefit and weight of item $i$, respectively.

| Item ($i$)  | Benefit ($b_i$) | Weight ($w_i$) |
| ----------- | ----------- | ----------- |
| hammer      | 8      | 5|
| wrench   | 3        | 7 |
| screwdriver  | 6 | 4        |
| towel   | 11  | 3 |

Let $W_{max} = 14$ be the maximum weight.

**Variables**

Let $x_i \in \{0,1\}$ (binary) represent whether or not we include item $i$ in the knapsack. For now, we will consider only being able to choose either none or one of each item.

**Objective and Constraints**

$$
\begin{equation} 
\begin{split}
\max_{x} \quad & \sum_{i\in{\mathcal{A}}}b_i x_i \\
\text{s.t.} \quad & \sum_{i\in{\mathcal{A}}}w_ix_i \leq W_{max} \\
& x_i \in \{0,1\}, \quad \forall i \in \mathcal{A}
\end{split}
\end{equation}
$$


**Pyomo**

Solve the knapsack problem given below using [HiGHS](https://highs.dev/) and answer the following questions:

1. Which items are acquired in the optimal solution?

2. Why does this solution make sense? (Write ~2 sentences.)

We use [HiGHS](https://highs.dev/), a modern open-source solver for linear and mixed-integer linear programs, which we call from Pyomo as `pyo.SolverFactory('appsi_highs')`. Earlier versions of this assignment used GLPK. HiGHS is faster, is actively developed, and installs anywhere with `pip install highspy`.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

# Add your solution here

model.display()

**Question Answers**

1. *Fill in here*

2. *Fill in here*

### 1.2 Knapsack example with improved printing

Complete the missing lines in the code below to produce formatted output: print the total weight, the value of the items selected (the objective), and the items acquired in the optimal solution. Note, the Pyomo value function should be used to get the floating point value of Pyomo modeling components (e.g., `print(value(model.x[i])`).

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

opt = pyo.SolverFactory("appsi_highs")
opt_success = opt.solve(model)
assert pyo.check_optimal_termination(opt_success), (
    f"Solve failed: status={opt_success.solver.status}, "
    f"termination={opt_success.solver.termination_condition}"
)

total_weight = sum(w[i] * pyo.value(model.x[i]) for i in A)
# Add your solution here

print("%12s %12s" % ("Item", "Selected"))
print("=========================")
for i in A:
    acquired = "No"
    # Add your solution here
print("-------------------------")

### 1.3 Changing data

Using your code from **Question 1.2**, if we were to increase the value of the wrench, at what point would it become selected as part of the optimal solution?

In [ ]:
# Add your solution here

**Question Answer**

*Fill in here*

### 1.4 Loading data from Excel

In the code above, the data is hardcoded at the top of the file. Instead of hardcoding the data, use Python to load the data from a different source. You may use Pandas to load data from 'knapsack_data.xlsx' into a dataframe. You will then need to write code to obtain a dictionary from the dataframe.

In [ ]:
df_items = pd.read_excel(
    "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/data/knapsack_data.xlsx", sheet_name="data", header=0, index_col=0
)
W_max = 14

A = df_items.index.tolist()
# Add your solution here

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

opt = pyo.SolverFactory("appsi_highs")
opt_success = opt.solve(model, tee=True)
assert pyo.check_optimal_termination(opt_success), (
    f"Solve failed: status={opt_success.solver.status}, "
    f"termination={opt_success.solver.termination_condition}"
)

total_weight = sum(w[i] * pyo.value(model.x[i]) for i in A)
print("Total Weight:", total_weight)
print("Total Benefit:", pyo.value(model.obj))

print("%12s %12s" % ("Item", "Selected"))
print("=========================")
for i in A:
    acquired = "No"
    if pyo.value(model.x[i]) >= 0.5:
        acquired = "Yes"
    print("%12s %12s" % (i, acquired))
print("-------------------------")

### 1.5 NLP vs. MIP

Solve the knapsack problem with IPOPT instead of HiGHS. Print the solution values for model.x. What happened? Why?

*Hint*: Switch `appsi_highs` to `ipopt` in the call to `SolverFactory`.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

# Add your solution here
opt_success = opt.solve(model, tee=True)
assert pyo.check_optimal_termination(opt_success), (
    f"Solve failed: status={opt_success.solver.status}, "
    f"termination={opt_success.solver.termination_condition}"
)

model.pprint()

**Question Answers**

*Fill in here*

## 2 More Pyomo Fundamentals

### 2.1 Knapsack problem with rules

Rules are important for defining indexed constraints, however, they can also be used for single (i.e. scalar) constraints. Reimplement the knapsack model from **Question 1.1** using rules for the objective and the constraints.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)


# Add your solution here

### 2.2 Integer formulation of the knapsack problem

Consider again the knapsack problem. Assume now that we can acquire multiple items of the same type. In this new formulation, $x_i$ is now an integer variable instead of a binary variable. One way to formulate this problem is as follows:

$$
\begin{equation} 
\begin{split}
\max_{x} \quad & \sum_{i\in{\mathcal{A}}}b_i x_i \\
\text{s.t.} \quad & \sum_{i\in{\mathcal{A}}}w_i x_i \leq W_{max} \\
 & x_i=\sum_{j=0}^Njq_{i,j}, \quad \forall i \in \mathcal{A} \\
 & 0 \leq x_i \leq N, \quad \forall i \in \mathcal{A} \\
 & q_{i,j} \in \{0,1\}, \quad \forall i \in \mathcal{A}, j \in \{0,...,N\}
\end{split}
\end{equation}
$$

One could optionally add the following constraint to select only one $q_{i,j}$ for each $i$, although it is not strictly necessary to yield an integer solution.
$$
\begin{equation}
\sum_{j=0}^N q_{i,j} = 1, \quad \forall i \in \mathcal{A}
\end{equation}
$$

Starting with your code from **Question 2.1**, implement this new formulation and solve. Is the solution surprising?

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14
N = range(6)  # create a list from 0-5

model = pyo.ConcreteModel()

model.x = pyo.Var(A)
model.q = pyo.Var(A, N, domain=pyo.Binary)


def obj_rule(m):
    return sum(b[i] * m.x[i] for i in A)


model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)


def weight_con_rule(m):
    return sum(w[i] * m.x[i] for i in A) <= W_max


model.weight_con = pyo.Constraint(rule=weight_con_rule)


# Add your solution here

**Question Answer**

*Fill in here*

## Some Advanced Pyomo Tricks

### Using the decorator notation for rules

Alternative notation for declaring and defining Pyomo components using decorators exists. Starting with the warehouse location problem code below, change the model to use the decorator notation.

In [ ]:
# warehouse_location.py: Warehouse location determination problem
model = pyo.ConcreteModel(name="(WL)")

W = ["Harlingen", "Memphis", "Ashland"]
C = ["NYC", "LA", "Chicago", "Houston"]
d = {
    ("Harlingen", "NYC"): 1956,
    ("Harlingen", "LA"): 1606,
    ("Harlingen", "Chicago"): 1410,
    ("Harlingen", "Houston"): 330,
    ("Memphis", "NYC"): 1096,
    ("Memphis", "LA"): 1792,
    ("Memphis", "Chicago"): 531,
    ("Memphis", "Houston"): 567,
    ("Ashland", "NYC"): 485,
    ("Ashland", "LA"): 2322,
    ("Ashland", "Chicago"): 324,
    ("Ashland", "Houston"): 1236,
}
P = 2

model.x = pyo.Var(W, C, bounds=(0, 1))
model.y = pyo.Var(W, domain=pyo.Binary)


@model.Objective()
def obj(m):
    return sum(d[w, c] * m.x[w, c] for w in W for c in C)


@model.Constraint(C)
def one_per_cust(m, c):
    return sum(m.x[w, c] for w in W) == 1


# Add your solution here
def warehouse_active(m, w, c):
    return m.x[w, c] <= m.y[w]


# Note: This is only split across cells because of a bug in nbpages (notebook/website software).
# There is no other reason to split your code across cells.

In [ ]:
# Add your solution here
def num_warehouses(m):
    return sum(m.y[w] for w in W) <= P


results = pyo.SolverFactory("appsi_highs").solve(model)
assert pyo.check_optimal_termination(results), (
    f"Solve failed: status={results.solver.status}, "
    f"termination={results.solver.termination_condition}"
)

model.y.pprint()
model.x.pprint()

### Changing parameter values

A parameter can be specified to be mutable. This tells Pyomo that the value of the parameter may change in the future, and allows the user to change the parameter value and resolve the problem without the need to rebuild the entire model each time. We will use this functionality to find a better solution to the knapsack problem. We would like to find when the wrench becomes valuable enough to be a part of the optimal solution. Create a Pyomo Parameter for the value of the items, make it mutable, and then write a loop that prints the solution for different wrench values.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)
# Add your solution here


def obj_rule(m):
    return sum(m.item_benefit[i] * m.x[i] for i in A)


model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)


def weight_rule(m):
    return sum(w[i] * m.x[i] for i in A) <= W_max


model.weight = pyo.Constraint(rule=weight_rule)

# You may instead use 'cbc' as the solver
opt = pyo.SolverFactory("appsi_highs")

for wrench_benefit in range(1, 11):
    model.item_benefit["wrench"] = wrench_benefit
    result_obj = opt.solve(model)
    assert pyo.check_optimal_termination(result_obj), (
        f"Solve failed: status={result_obj.solver.status}, "
        f"termination={result_obj.solver.termination_condition}"
    )

    # Add your solution here

### Integer cuts

Often, it can be important to find not only the "best" solution, but a number of solutions that are equally optimal, or close to optimal. For discrete optimization problems, this can be done using something known as an integer cut. Consider again the knapsack problem where the choice of which items to select is a discrete variable $x_i \forall i \in A$. Let $x_i^*$ be a particular set of $x$ values we want to remove from the feasible solution space. We define an integer cut using two sets. The first set $S_0$ contains the indices for those variables whose current solution is 0, and the second set $S_1$ consists of indices for those variables whose current solution is 1. Given these two sets, an integer cut constraint that would prevent such a solution from appearing again is defined by,

$\sum_{i \in S_0}x[i] + \sum_{i \in S_1}(1-x_i) \geq 1$

Write a loop that solves the problem 5 times, adding an integer cut to remove the previous solution, and printing the value of the objective function and the solution at each iteration of the loop.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)


def obj_rule(m):
    return sum(b[i] * m.x[i] for i in A)


model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)


def weight_con_rule(m):
    return sum(w[i] * m.x[i] for i in A) <= W_max


model.weight_con = pyo.Constraint(rule=weight_con_rule)

# You may instead use 'cbc' as the solver
opt = pyo.SolverFactory("appsi_highs")

# create the ConstraintList to hold the integer cuts
model.int_cuts = pyo.ConstraintList()

# Add your solution here

### Putting it all together: Lot sizing example (Bynum et al., 2021)

We will now write a complete model from scratch using a well-known multi-period optimization problem for optimal lot-sizing adapted from Haugen et al. (2001) shown below.

$\min \sum_{t \in T}c_ty_t+h_t^+I_t^+ +h_t^-I_t^-$

s.t. $I_t=I_{t-1}+X_t-d_t, \forall t \in T$

$I_t=I_t^+-I_t^-, \forall t \in T$

$X_t \leq Py_t, \forall t \in T$

$X_t, I_t^+, I_t^- \geq 0, \forall t \in T$

$y_t \in \{0,1\}, \forall t \in T$

Our goal is to find the optimal production $X_t$ given known demands $d_t$, fixed cost $c_t$ associated with active production in a particular time period, an inventory holding cost $h_t^+$ and a shortage cost $h_t^-$ (cost of keeping a backlog) of orders. The variable $y_t$ (binary) determines if we produce in time $t$ or not, and $I_t^+$ represents inventory that we are storing across time period $t$, while $I_t^-$ represents the magnitude of the backlog. Note that $X_t \leq Py_t$ is a constraint that only allows production in time period $t$ if the indicator variable $y_t$=1.

Write a Pyomo model for this problem and solve it using HiGHS, i.e., `pyo.SolverFactory('appsi_highs')`, (or cbc) using the data provided below.

|Parameter|Description|Value|
|---|---|---|
|$c$|fixed cost of production|4.6|
|$I_0^+$|initial value of positive inventory|5.0|
|$I_0^-$|initial value of backlogged orders|0.0|
|$h^+$|cost (per unit) of holding inventory|0.7|
|$h^-$|shortage cost (per unit)|1.2|
|$P$|maximum production amount (big-M value)|5|
|$d$|demand|[5, 7, 6.2, 3.1, 1.7]|

**Reference**: Bynum, M. L., Hackebeil, G. A., Hart, W. E., Laird, C. D., Nicholson, B. L., Siirola, J. D., Watson, J.-P., and Woodruff, D. L. *Pyomo — Optimization Modeling in Python*, Third Edition. Springer Optimization and Its Applications, Vol. 67, 2021. (§8.6, p. 117)

In [ ]:
model = pyo.ConcreteModel()
model.T = pyo.RangeSet(5)  # time periods

i0 = 5.0  # initial inventory
c = 4.6  # setup cost
h_pos = 0.7  # inventory holding cost
h_neg = 1.2  # shortage cost
P = 5.0  # maximum production amount

# demand during period t
d = {1: 5.0, 2: 7.0, 3: 6.2, 4: 3.1, 5: 1.7}

# Add your solution here

# solve the problem
# You may instead use 'cbc' as the solver
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(model)
assert pyo.check_optimal_termination(results), (
    f"Solve failed: status={results.solver.status}, "
    f"termination={results.solver.termination_condition}"
)

# print the results
for t in model.T:
    print("Period: {0}, Prod. Amount: {1}".format(t, pyo.value(model.x[t])))